# Running a function on Fleets

This tutorial walks through uploading and running a function on **Fleets**, the engine behind IBM Quantum Functions (formerly Qiskit Functions), IBM's hosted managed-workflows service (built on IBM Cloud Code Engine). See [Concepts: Runners & Naming](https://qiskit.github.io/qiskit-serverless/concepts/index.html) for how Fleets relates to Ray and how this SDK's naming maps to the official terminology.

> &#x26A0; This notebook runs against a **local mock of Code Engine**, not a real cloud account, so it's free and requires no IBM Cloud credentials. It uses the same gateway, scheduler, and client SDK as production — only Code Engine itself is swapped for a mock that runs your job's command in a local container instead of a real cloud fleet. See [Fleets integration tests](https://qiskit.github.io/qiskit-serverless/local_setup/fleets_integration_tests.html) for how the stack works. Against a real IBM Cloud account, the client code below is identical except you'd use `IBMServerlessClient` with your IBM Quantum Platform credentials instead of `ServerlessClient` pointed at `localhost` — see [IBM Quantum: Managed workflows](https://quantum.cloud.ibm.com/docs/en/managed-workflows) for that path.

The API is the same `QiskitFunction`/`ServerlessClient` shape as the Ray tutorials — the only difference is `runner="fleets"` (which is also the default if you omit it).

## 1. Start the local Fleets stack

From the repository root:

```bash
# Build the fleets runtime base image (the fleet-worker builds FROM it).
make build-fleet-node

# Start the mock stack: gateway, scheduler, postgres, MinIO, fleet-worker.
docker compose -f docker-compose-fleets-test.yaml up --build -d

# Wait until the gateway is ready.
curl --retry 30 --retry-delay 2 --retry-all-errors --fail http://localhost:8000/liveness/
```

When you're done, tear it down with `docker compose -f docker-compose-fleets-test.yaml down -v`.

## 2. Write the function

Save the following as [function.py](./source_files/function.py):

```python
from qiskit_serverless import get_arguments, get_logger, save_result

logger = get_logger()

arguments = get_arguments()
name = arguments.get("name", "world")

logger.info("Running function... name=%s", name)
save_result({"greeting": f"Hello, {name}!"})
logger.info("Completed running function.")
```

This is a plain function: no Ray, no `@ray.remote`. A Fleets job runs your entrypoint as a regular Python process in the container.

## 3. Configure the client

This mock stack uses the same mock-token auth as the other local tutorials, on the same gateway port.

In [ ]:
from qiskit_serverless import ServerlessClient
import os

client = ServerlessClient(
    token=os.environ.get("GATEWAY_TOKEN", "awesome_token"),
    instance=os.environ.get("GATEWAY_INSTANCE", "an_awesome_crn"),
    host=os.environ.get("GATEWAY_HOST", "http://localhost:8000"),
)

client

## 4. Create, upload, and run the function

`runner="fleets"` is the default for `QiskitFunction` — passed explicitly here for clarity.

In [ ]:
from qiskit_serverless import QiskitFunction

function = QiskitFunction(
    title="fleets-hello-world",
    entrypoint="function.py",
    working_dir="./source_files/",
    runner="fleets",  # this is also the default
)

client.upload(function)

Fetch the uploaded function by title and run it, same as any other function:

In [ ]:
my_function = client.get("fleets-hello-world")

job = my_function.run(name="Fleets")
job

Check status and wait for the result, exactly as with Ray-backed functions:

In [ ]:
job.status()

In [ ]:
job.result()

Logs work the same way too:

In [ ]:
print(job.logs())

## Next steps

- For a function with arguments and more structured results, see the [basic tutorials](https://qiskit.github.io/qiskit-serverless/function_features/basic/index.html) (Ray-based, but the argument-passing API is identical).
- For running a **provider** (image-based) function on Fleets, see [Deploy a Custom Function Image](https://qiskit.github.io/qiskit-serverless/custom_images/deploy_custom_image_function.html), which covers both engines. Note a custom provider image can't be exercised against this local mock stack — it only has a documented path against a real environment.
- For the production path against IBM's hosted service, see [IBM Quantum: Managed workflows](https://quantum.cloud.ibm.com/docs/en/managed-workflows).